<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=344809700" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 16 SESSION: FOLD 2, CUSTOM CNN + EFFICIENTNETB0 (loads locked fold file, never regenerates) =====
# First touch on fold 2. This session ESTABLISHES fold 2's split sizes and class weight span,
# the same way the original fold 1 session did. The later fold 2 mob/res session will assert
# reproduction against whatever prints here.

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, roc_auc_score, accuracy_score,
                              f1_score, recall_score, confusion_matrix)

# ---------- LOAD LOCKED FOLD ASSIGNMENTS, DO NOT REGENERATE ----------
FOLD_CSV_PATH = '/kaggle/input/datasets/asivakumarnair/drcvfoldassignments/dr_cv_fold_assignments.csv'

GRADES, NUM_CLASSES = ['0','1','2','3','4'], 5
IMG_SIZE, BATCH_SIZE = 224, 32
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, CUSTOM_LR, EARLYSTOP_PAT = 10, 1e-3, 1e-5, 1e-3, 7
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
CURRENT_FOLD = 2

pooled = pd.read_csv(FOLD_CSV_PATH)
pooled['grade'] = pooled['grade'].astype(str)
print(f"Loaded {len(pooled):,} rows (expect 9,068)")
assert len(pooled) == 9068, "Row count mismatch, wrong file or corrupted upload"
assert pooled['fold'].nunique() == 5, "Fold file does not have 5 folds"

test_df   = pooled[pooled.fold == CURRENT_FOLD].reset_index(drop=True)
remainder = pooled[pooled.fold != CURRENT_FOLD].reset_index(drop=True)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_group_level_inner(df, label_col='grade', val_frac=0.15, rs=SEED, tag=""):
    grouped = df.groupby('group_id')[label_col].agg(lambda s: s.value_counts().index[0]).reset_index()
    g_tr, g_va = safe_split(grouped, label_col, val_frac, rs, tag=tag)
    pick = lambda ids: df[df['group_id'].isin(ids['group_id'])]
    return pick(g_tr), pick(g_va)

train_parts, val_parts = [], []
for src in ['aptos', 'eyepacs', 'messidor']:
    sub = remainder[remainder.source == src]
    tr_s, va_s = split_group_level_inner(sub, tag=f"fold{CURRENT_FOLD}-{src}-inner")
    train_parts.append(tr_s); val_parts.append(va_s)
train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)

print(f"\nFold {CURRENT_FOLD}: Train {len(train_df):,} ({len(train_df)/len(pooled)*100:.1f}%) | "
      f"Val {len(val_df):,} ({len(val_df)/len(pooled)*100:.1f}%) | "
      f"Test {len(test_df):,} ({len(test_df)/len(pooled)*100:.1f}%)")
print("^^ RECORD THESE NUMBERS. The fold 2 mob/res session will assert reproduction against them.")

for src in ['aptos', 'eyepacs', 'messidor']:
    tr_g = set(train_df[train_df.source==src]['group_id'])
    va_g = set(val_df[val_df.source==src]['group_id'])
    te_g = set(test_df[test_df.source==src]['group_id'])
    ok = tr_g.isdisjoint(va_g) and tr_g.isdisjoint(te_g) and va_g.isdisjoint(te_g)
    print(f"  {src}: train/val/test group-disjoint = {ok}")
    assert ok, f"LEAKAGE in fold {CURRENT_FOLD}, source {src}"
print(f"Fold {CURRENT_FOLD} leakage check: PASS")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['grade'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}
print(f"Fold {CURRENT_FOLD} class_weight (fresh from this fold's train set):",
      {c: round(w,3) for c,w in zip(cls,cw)}, f"| span {cw.max()/cw.min():.1f}x")
print("^^ RECORD THIS SPAN too, same reproduction requirement.")

def make_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=GRADES, color_mode='rgb')
    return (train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common),
            eval_idg.flow_from_dataframe(val_df, shuffle=False, **common),
            eval_idg.flow_from_dataframe(test_df, shuffle=False, **common))

def build_custom_cnn(num_classes=5, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                        Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes=NUM_CLASSES):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp) > 0 else np.nan)
    return np.nanmean(specs)

def full_test_metrics(model, te_gen):
    y_prob = model.predict(te_gen, verbose=0)
    y_true = np.asarray(te_gen.classes)
    y_pred = y_prob.argmax(axis=1)
    try:
        auc = roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr')
    except ValueError:
        auc = np.nan
    return dict(qwk=cohen_kappa_score(y_true, y_pred, weights='quadratic'), macro_auc=auc,
                accuracy=accuracy_score(y_true, y_pred),
                macro_f1=f1_score(y_true, y_pred, average='macro'),
                macro_sensitivity=recall_score(y_true, y_pred, average='macro'),
                macro_specificity=macro_specificity(y_true, y_pred), n_test=len(y_true)), y_true, y_pred, y_prob

def train_and_evaluate(arch_code, build_fn, preprocess_fn, is_pretrained):
    tr, va, te = make_gens(preprocess_fn)
    auc_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_aucbest.keras'
    acc_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_accbest.keras'

    if is_pretrained:
        model, base = build_fn()
        base.trainable = False
        model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
        print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
        model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=CLASS_WEIGHT, verbose=1,
                  callbacks=[CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=False)])
        base.trainable = True
        model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
        print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: PHASE 2 (full fine-tune, dual checkpoint) =====")
        hist = model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT, verbose=1,
                  callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                             ModelCheckpoint(auc_path, monitor='val_auc', mode='max', save_best_only=True),
                             ModelCheckpoint(acc_path, monitor='val_accuracy', mode='max', save_best_only=True),
                             CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=True)])
    else:
        model = build_fn()
        model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
        print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: single phase, dual checkpoint =====")
        hist = model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT, verbose=1,
                  callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                             ModelCheckpoint(auc_path, monitor='val_auc', mode='max', save_best_only=True),
                             ModelCheckpoint(acc_path, monitor='val_accuracy', mode='max', save_best_only=True),
                             CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=False)])

    live_metrics, y_true, y_pred, y_prob = full_test_metrics(model, te)
    print(f"\nLive (in-memory) test metrics, auc-selected: {live_metrics}")
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_preds.npz',
             y_true=y_true, y_pred=y_pred, y_prob=y_prob,
             source=test_df['source'].values, group_id=test_df['group_id'].values)

    reloaded_auc_model = load_model(auc_path)
    reloaded_auc_metrics, _, _, _ = full_test_metrics(reloaded_auc_model, te)
    match = abs(live_metrics['macro_auc'] - reloaded_auc_metrics['macro_auc']) < 1e-3
    print(f"Reloaded auc-checkpoint macro_auc={reloaded_auc_metrics['macro_auc']:.4f} vs live={live_metrics['macro_auc']:.4f}, match={match}")
    assert match, "AUC-CHECKPOINT MISMATCH, invalid provenance, discard this result"
    del reloaded_auc_model

    best_val_acc_seen = max(hist.history['val_accuracy'])
    reloaded_acc_model = load_model(acc_path)
    reloaded_val_acc = reloaded_acc_model.evaluate(va, verbose=0)[1]
    match_acc = abs(best_val_acc_seen - reloaded_val_acc) < 1e-3
    print(f"Accuracy-checkpoint provenance: best val_accuracy seen during training={best_val_acc_seen:.4f} "
          f"vs reloaded val_accuracy={reloaded_val_acc:.4f}, match={match_acc}")
    assert match_acc, "ACCURACY-CHECKPOINT MISMATCH, invalid provenance, discard this result"
    reloaded_acc_metrics, _, _, _ = full_test_metrics(reloaded_acc_model, te)
    del reloaded_acc_model
    del model; import gc; gc.collect(); tf.keras.backend.clear_session()

    same_checkpoint = abs(reloaded_auc_metrics['macro_auc'] - reloaded_acc_metrics['macro_auc']) < 1e-6
    print(f"Monitors agreed on the same epoch: {same_checkpoint}")

    rows = [
        {'fold': CURRENT_FOLD, 'arch': arch_code, 'selected_by': 'val_auc', **reloaded_auc_metrics},
        {'fold': CURRENT_FOLD, 'arch': arch_code, 'selected_by': 'val_accuracy', **reloaded_acc_metrics},
    ]
    out_df = pd.DataFrame(rows)
    out_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_results.csv'
    out_df.to_csv(out_path, index=False)
    print(f"\nSaved {out_path}")
    print(out_df.round(4).to_string(index=False))
    return out_df

# ---- CUSTOM CNN ----
custom_results = train_and_evaluate('custom', build_custom_cnn, None, is_pretrained=False)

# ---- EFFICIENTNETB0 ----
eff_results = train_and_evaluate('eff', lambda: build_pretrained(EfficientNetB0), eff_pre, is_pretrained=True)

print(f"\n\n{'='*20} FOLD {CURRENT_FOLD}: CUSTOM + EFF DONE (2 of 4) {'='*20}")
print(pd.concat([custom_results, eff_results], ignore_index=True).round(4).to_string(index=False))
print(f"\nDownload individually: cv_f{CURRENT_FOLD}_custom_results.csv, cv_f{CURRENT_FOLD}_eff_results.csv")
print(f"Still needed for fold {CURRENT_FOLD}: mob, res. Do not start fold 3 until all 4 exist.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-25 08:40:33.329784: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787647233.351830      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787647233.359202      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787647233.377393      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787647233.377411      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787647233.377413      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Loaded 9,068 rows (expect 9,068)

Fold 2: Train 6,166 (68.0%) | Val 1,090 (12.0%) | Test 1,812 (20.0%)
^^ RECORD THESE NUMBERS. The fold 2 mob/res session will assert reproduction against them.
  aptos: train/val/test group-disjoint = True
  eyepacs: train/val/test group-disjoint = True
  messidor: train/val/test group-disjoint = True
Fold 2 leakage check: PASS
Fold 2 class_weight (fresh from this fold's train set): {np.str_('0'): np.float64(0.326), np.str_('1'): np.float64(2.055), np.str_('2'): np.float64(0.966), np.str_('3'): np.float64(5.096), np.str_('4'): np.float64(4.601)} | span 15.6x
^^ RECORD THIS SPAN too, same reproduction requirement.
Found 6166 validated image filenames belonging to 5 classes.
Found 1090 validated image filenames belonging to 5 classes.
Found 1812 validated image filenames belonging to 5 classes.


I0000 00:00:1787647261.050438      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787647261.056329      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



===== Fold 2, custom: single phase, dual checkpoint =====
Epoch 1/60


...

193/193 [==============================] - 525s 3s/step - loss: 1.6291 - accuracy: 0.3394 - auc: 0.6459 - val_loss: 2.7240 - val_accuracy: 0.0899 - val_auc: 0.1953
Epoch 2/60
193/193 [==============================] - 415s 2s/step - loss: 1.5491 - accuracy: 0.4267 - auc: 0.7104 - val_loss: 2.2365 - val_accuracy: 0.0725 - val_auc: 0.2079
Epoch 3/60
193/193 [==============================] - 415s 2s/step - loss: 1.5058 - accuracy: 0.4643 - auc: 0.7490 - val_loss: 1.2587 - val_accuracy: 0.5908 - val_auc: 0.8113
Epoch 4/60
193/193 [==============================] - 410s 2s/step - loss: 1.5067 - accuracy: 0.4873 - auc: 0.7589 - val_loss: 1.2428 - val_accuracy: 0.6294 - val_auc: 0.8143
Epoch 5/60
193/193 [==============================] - 417s 2s/step - loss: 1.4784 - accuracy: 0.4971 - auc: 0.7641 - val_loss: 1.1849 - val_accuracy: 0.6468 - val_auc: 0.8385
Epoch 6/60
193/193 [==============================] - 415s 2s/step - loss: 1.4719 - accuracy: 0.4985 - auc: 0.7768 - val_loss: 1.3018 - 

E0000 00:00:1787655692.486224      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


193/193 [==============================] - 398s 2s/step - loss: 1.3759 - accuracy: 0.4715 - auc: 0.7897 - val_loss: 1.2364 - val_accuracy: 0.3486 - val_auc: 0.7860
Epoch 2/10
193/193 [==============================] - 395s 2s/step - loss: 1.2115 - accuracy: 0.5490 - auc: 0.8421 - val_loss: 1.2304 - val_accuracy: 0.4294 - val_auc: 0.7969
Epoch 3/10
193/193 [==============================] - 393s 2s/step - loss: 1.1785 - accuracy: 0.5709 - auc: 0.8532 - val_loss: 1.1030 - val_accuracy: 0.5083 - val_auc: 0.8406
Epoch 4/10
193/193 [==============================] - 396s 2s/step - loss: 1.1289 - accuracy: 0.5911 - auc: 0.8649 - val_loss: 1.1484 - val_accuracy: 0.4339 - val_auc: 0.8161
Epoch 5/10
193/193 [==============================] - 392s 2s/step - loss: 1.0995 - accuracy: 0.5864 - auc: 0.8695 - val_loss: 1.4233 - val_accuracy: 0.3073 - val_auc: 0.7189
Epoch 6/10
193/193 [==============================] - 386s 2s/step - loss: 1.1034 - accuracy: 0.5958 - auc: 0.8718 - val_loss: 1.2471 - 

E0000 00:00:1787659619.898727      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


193/193 [==============================] - 478s 2s/step - loss: 1.9110 - accuracy: 0.4460 - auc: 0.7393 - val_loss: 0.9877 - val_accuracy: 0.6661 - val_auc: 0.8841
Epoch 2/60
193/193 [==============================] - 423s 2s/step - loss: 1.5173 - accuracy: 0.4693 - auc: 0.7721 - val_loss: 0.9964 - val_accuracy: 0.6505 - val_auc: 0.8798
Epoch 3/60
193/193 [==============================] - 421s 2s/step - loss: 1.3960 - accuracy: 0.4909 - auc: 0.7954 - val_loss: 1.0042 - val_accuracy: 0.6147 - val_auc: 0.8745
Epoch 4/60
193/193 [==============================] - 424s 2s/step - loss: 1.3196 - accuracy: 0.5097 - auc: 0.8062 - val_loss: 0.9959 - val_accuracy: 0.6220 - val_auc: 0.8753
Epoch 5/60
193/193 [==============================] - 432s 2s/step - loss: 1.2704 - accuracy: 0.5114 - auc: 0.8168 - val_loss: 0.9804 - val_accuracy: 0.6266 - val_auc: 0.8789
Epoch 6/60
193/193 [==============================] - 428s 2s/step - loss: 1.2068 - accuracy: 0.5341 - auc: 0.8327 - val_loss: 0.9637 - 